# Phishing URL Detection – Model Training & Comparison Notebook

This notebook walks through the **complete ML training pipeline** for the phishing URL detector:

1. Load and explore the dataset
2. Feature analysis & importance
3. Train multiple models (GBC, XGBoost, RandomForest, LightGBM)
4. Cross-validation with stratified k-fold
5. Hyperparameter tuning for best accuracy
6. Performance metrics (Accuracy, Precision, Recall, F1, ROC-AUC, Confusion Matrix)
7. Save the best model to `pickle/model.pkl`

**Labels:** `1` = Legitimate/Safe, `-1` = Phishing/Unsafe

## 1. Setup & Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
import json
from pathlib import Path

from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, classification_report
)
from sklearn.preprocessing import LabelEncoder

try:
    import xgboost as xgb
    HAS_XGB = True
    print('XGBoost available:', xgb.__version__)
except ImportError:
    HAS_XGB = False
    print('XGBoost not installed – skipping')

try:
    import lightgbm as lgb
    HAS_LGB = True
    print('LightGBM available:', lgb.__version__)
except ImportError:
    HAS_LGB = False
    print('LightGBM not installed – skipping')

# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

%matplotlib inline
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_theme(style='whitegrid')
print('Setup complete!')

## 2. Load Dataset

In [ ]:
FEATURE_COLUMNS = [
    'UsingIP', 'LongURL', 'ShortURL', 'Symbol@', 'Redirecting//',
    'PrefixSuffix-', 'SubDomains', 'HTTPS', 'DomainRegLen', 'Favicon',
    'NonStdPort', 'HTTPSDomainURL', 'RequestURL', 'AnchorURL',
    'LinksInScriptTags', 'ServerFormHandler', 'InfoEmail', 'AbnormalURL',
    'WebsiteForwarding', 'StatusBarCust', 'DisableRightClick',
    'UsingPopupWindow', 'IframeRedirection', 'AgeofDomain', 'DNSRecording',
    'WebsiteTraffic', 'PageRank', 'GoogleIndex', 'LinksPointingToPage',
    'StatsReport',
]
TARGET_COLUMN = 'class'

# Try the original dataset first; fall back to generated synthetic data
data_path = 'phishing.csv'
if not Path(data_path).exists():
    print(f"'{data_path}' not found. Generating synthetic dataset...")
    import subprocess
    subprocess.run(['python', 'generate_sample_data.py', '--out', 'dataset.csv', '--samples', '2000'])
    data_path = 'dataset.csv'

df = pd.read_csv(data_path, index_col=0)
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

print(f'Dataset: {data_path}')
print(f'Shape: {df.shape}')
print(f'\nClass distribution:')
print(df[TARGET_COLUMN].value_counts().rename({1: 'Legitimate (1)', -1: 'Phishing (-1)'}))
df.head()

## 3. Exploratory Data Analysis

In [ ]:
# Class balance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

counts = df[TARGET_COLUMN].value_counts()
labels = ['Legitimate (1)', 'Phishing (-1)']
colors = ['#2196F3', '#F44336']

axes[0].pie(counts.values, labels=labels, colors=colors, autopct='%1.1f%%',
            startangle=90, shadow=True)
axes[0].set_title('Class Distribution', fontsize=14, fontweight='bold')

axes[1].bar(['Legitimate', 'Phishing'], counts.values, color=colors, edgecolor='black', linewidth=0.5)
axes[1].set_title('Sample Count per Class', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[1].text(i, v + 50, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature value distributions
fig, axes = plt.subplots(6, 5, figsize=(22, 24))
axes = axes.flatten()

for i, col in enumerate(FEATURE_COLUMNS):
    ax = axes[i]
    for label, color in [(-1, '#F44336'), (1, '#2196F3')]:
        subset = df[df[TARGET_COLUMN] == label][col].value_counts().sort_index()
        ax.bar(subset.index + (0.2 if label == 1 else -0.2), subset.values,
               width=0.4, color=color, alpha=0.7,
               label='Legitimate' if label == 1 else 'Phishing')
    ax.set_title(col, fontsize=9, fontweight='bold')
    ax.set_xticks([-1, 0, 1])
    if i == 0:
        ax.legend(fontsize=8)

# Hide empty subplots
for j in range(len(FEATURE_COLUMNS), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Value Distributions by Class', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Feature correlation heatmap
plt.figure(figsize=(18, 14))
corr = df[FEATURE_COLUMNS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlGn', center=0,
            annot=False, linewidths=0.5, vmin=-1, vmax=1,
            cbar_kws={'label': 'Pearson Correlation'})
plt.title('Feature Correlation Matrix', fontsize=16, fontweight='bold')
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(fontsize=9)
plt.tight_layout()
plt.show()

## 4. Prepare Data for Training

In [ ]:
X = df[FEATURE_COLUMNS].values
y_raw = df[TARGET_COLUMN].values

# Encode labels: -1 -> 0, 1 -> 1  (required by XGBoost/LightGBM)
le = LabelEncoder()
y = le.fit_transform(y_raw)
print(f'Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f'Train: {len(X_train):,}  |  Test: {len(X_test):,}')

## 5. Train Multiple Models

In [ ]:
import time

models = {
    'GradientBoosting': GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=5,
        random_state=RANDOM_STATE,
    ),
    'RandomForest': RandomForestClassifier(
        n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1,
    ),
}

if HAS_XGB:
    models['XGBoost'] = xgb.XGBClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=6,
        use_label_encoder=False, eval_metric='logloss',
        random_state=RANDOM_STATE, n_jobs=-1,
    )

if HAS_LGB:
    models['LightGBM'] = lgb.LGBMClassifier(
        n_estimators=200, learning_rate=0.05, max_depth=-1,
        random_state=RANDOM_STATE, n_jobs=1, verbose=-1,
    )

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
results = {}

for name, model in models.items():
    print(f'Training {name}...')
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    # Cross-validation (n_jobs=1 avoids thread conflicts with LightGBM/XGB)
    cv_acc = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=1)
    cv_f1  = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=1)

    y_pred  = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    results[name] = {
        'model':            model,
        'train_time_s':     round(train_time, 2),
        'cv_acc_mean':      round(cv_acc.mean(), 4),
        'cv_acc_std':       round(cv_acc.std(),  4),
        'test_accuracy':    round(accuracy_score(y_test, y_pred), 4),
        'test_precision':   round(precision_score(y_test, y_pred, average='weighted', zero_division=0), 4),
        'test_recall':      round(recall_score(y_test, y_pred,    average='weighted', zero_division=0), 4),
        'test_f1':          round(f1_score(y_test, y_pred,        average='weighted', zero_division=0), 4),
        'test_roc_auc':     round(roc_auc_score(y_test, y_proba), 4),
        'y_pred':           y_pred,
        'y_proba':          y_proba,
    }
    print(f'  Train time : {train_time:.1f}s  |  CV Acc: {cv_acc.mean():.4f} ± {cv_acc.std():.4f}  |  Test Acc: {results[name]["test_accuracy"]:.4f}\n')

print('All models trained!')

## 6. Model Comparison

In [ ]:
summary = pd.DataFrame([
    {
        'Model': name,
        'Train Time (s)': r['train_time_s'],
        'CV Accuracy': r['cv_acc_mean'],
        'CV Std': r['cv_acc_std'],
        'Test Accuracy': r['test_accuracy'],
        'Precision': r['test_precision'],
        'Recall': r['test_recall'],
        'F1 Score': r['test_f1'],
        'ROC-AUC': r['test_roc_auc'],
    }
    for name, r in results.items()
]).set_index('Model')

display(summary.style
    .highlight_max(subset=['CV Accuracy', 'Test Accuracy', 'F1 Score', 'ROC-AUC'], color='#c8f7c5')
    .highlight_min(subset=['Train Time (s)'], color='#c8f7c5')
    .format({
        'CV Accuracy': '{:.4f}', 'CV Std': '±{:.4f}',
        'Test Accuracy': '{:.4f}', 'Precision': '{:.4f}',
        'Recall': '{:.4f}', 'F1 Score': '{:.4f}', 'ROC-AUC': '{:.4f}',
    })
)

In [ ]:
metrics_to_plot = ['Test Accuracy', 'F1 Score', 'ROC-AUC']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette = sns.color_palette('husl', len(results))

for ax, metric in zip(axes, metrics_to_plot):
    bars = ax.bar(summary.index, summary[metric], color=palette, edgecolor='black', linewidth=0.5)
    ax.set_title(metric, fontsize=13, fontweight='bold')
    ax.set_ylim(max(0.85, summary[metric].min() - 0.02), 1.01)
    ax.set_ylabel('Score')
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, summary[metric]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('Model Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Confusion Matrices

In [ ]:
n_models = len(results)
fig, axes = plt.subplots(1, n_models, figsize=(5 * n_models, 4))
if n_models == 1:
    axes = [axes]

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Phishing', 'Legitimate'],
    )
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nAcc={r["test_accuracy"]:.4f}', fontweight='bold')

plt.suptitle('Confusion Matrices (Test Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 8. ROC Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
palette = sns.color_palette('husl', len(results))

for (name, r), color in zip(results.items(), palette):
    RocCurveDisplay.from_predictions(
        y_test, r['y_proba'],
        name=f"{name} (AUC={r['test_roc_auc']:.4f})",
        ax=ax, color=color,
    )

ax.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
ax.set_title('ROC Curves – All Models', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
# Collect feature importances from tree-based models
importance_df = pd.DataFrame(index=FEATURE_COLUMNS)

for name, r in results.items():
    model = r['model']
    if hasattr(model, 'feature_importances_'):
        importance_df[name] = model.feature_importances_

if importance_df.empty:
    print('No tree-based models with feature_importances_ available.')
else:
    importance_df['Mean Importance'] = importance_df.mean(axis=1)
    importance_df = importance_df.sort_values('Mean Importance', ascending=True)

    fig, ax = plt.subplots(figsize=(10, 10))
    colors = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(FEATURE_COLUMNS)))
    ax.barh(importance_df.index, importance_df['Mean Importance'], color=colors, edgecolor='black', linewidth=0.3)
    ax.set_title('Mean Feature Importance Across Models', fontsize=14, fontweight='bold')
    ax.set_xlabel('Mean Importance')
    plt.tight_layout()
    plt.show()

    print('Top 10 most important features:')
    print(importance_df[['Mean Importance']].tail(10).sort_values('Mean Importance', ascending=False).to_string())

## 10. Hyperparameter Tuning (Best Model)

In [ ]:
best_name = max(results, key=lambda n: results[n]['test_accuracy'])
print(f'Best model before tuning: {best_name}  (acc={results[best_name]["test_accuracy"]:.4f})')

PARAM_GRIDS = {
    'GradientBoosting': {
        'n_estimators': [100, 200, 300],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 5, 7],
    },
    'RandomForest': {
        'n_estimators': [100, 200, 300],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5],
    },
    'XGBoost': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'max_depth': [4, 6],
    },
    'LightGBM': {
        'n_estimators': [100, 200],
        'learning_rate': [0.05, 0.1],
        'num_leaves': [31, 63],
    },
}

if best_name in PARAM_GRIDS:
    print(f'Running GridSearchCV for {best_name}...')
    cv_search = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    grid_search = GridSearchCV(
        results[best_name]['model'],
        PARAM_GRIDS[best_name],
        cv=cv_search, scoring='accuracy',
        n_jobs=-1, verbose=1, refit=True,
    )
    grid_search.fit(X_train, y_train)
    best_model = grid_search.best_estimator_
    print(f'\nBest params: {grid_search.best_params_}')
    print(f'Best CV accuracy: {grid_search.best_score_:.4f}')

    y_pred_tuned  = best_model.predict(X_test)
    y_proba_tuned = best_model.predict_proba(X_test)[:, 1]
    tuned_acc = accuracy_score(y_test, y_pred_tuned)
    tuned_f1  = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)
    tuned_auc = roc_auc_score(y_test, y_proba_tuned)
    print(f'\nTuned test accuracy: {tuned_acc:.4f}  (was {results[best_name]["test_accuracy"]:.4f})')
    print(f'Tuned F1: {tuned_f1:.4f}  |  ROC-AUC: {tuned_auc:.4f}')
else:
    best_model = results[best_name]['model']
    print('No param grid defined – using default best model.')

## 11. Final Classification Report

In [ ]:
y_pred_final = best_model.predict(X_test)
print(f'Final Model: {best_name}')
print('=' * 50)
print(classification_report(
    le.inverse_transform(y_test),
    le.inverse_transform(y_pred_final),
    target_names=['Phishing (-1)', 'Legitimate (1)'],
    zero_division=0,
))

## 12. Save Best Model

In [ ]:
from model_utils import LabelDecodingWrapper

Path('pickle').mkdir(exist_ok=True)

# Wrap model so app.py (which expects {-1, 1} labels) still works
wrapped_model = LabelDecodingWrapper(best_model, le)

with open('pickle/model.pkl', 'wb') as fh:
    pickle.dump(wrapped_model, fh)

# Quick sanity check
with open('pickle/model.pkl', 'rb') as fh:
    loaded_model = pickle.load(fh)

sample = X_test[0].reshape(1, -1)
pred  = loaded_model.predict(sample)
proba = loaded_model.predict_proba(sample)
print(f'Model saved to pickle/model.pkl')
print(f'Sample prediction: {pred}  (1=legitimate, -1=phishing)')
print(f'Probabilities: phishing={proba[0,0]:.3f}  legitimate={proba[0,1]:.3f}')
print('Model is app.py compatible!')

## Summary

| Metric | Value |
|--------|-------|
| Best Model | See results above |
| Test Accuracy | See results above |
| ROC-AUC | See results above |

### Next Steps
- Collect more real phishing/legitimate URLs and re-run `train.py`
- Experiment with additional features (WHOIS age, TLD analysis, entropy)
- Try stacking/blending the top models for even higher accuracy
- Deploy the updated `pickle/model.pkl` in the Flask app